## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101_raw")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID Cleanup

In [0]:
df = df.withColumn(
    "CID", F.regexp_replace(col("CID"),"-", "")
    )

## Country Normalization

In [0]:
df = df.withColumn(
    "CNTRY",
    F.when(col("CNTRY").isin("US", "USA"), "United States")
    .when(col("CNTRY") == "DE", "Germany")
    .when((col("CNTRY") == "") | col("CNTRY").isNull(), "n/a")
    .otherwise(col("CNTRY"))
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "CNTRY": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity Check of Dataframe

In [0]:
display(df)

# Writting Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

## Sanity Check of Silver Table

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_customer_location
LIMIT 10